# Lane Detection using Classical Computer Vision## CV Assignment 1 - Problem 2### Course: Computer Vision (AIMLCZG525) | Semester 2, 2025-26---**Group Members:**1. [Your Name] - [Your ID]2. [Member 2] - [ID]3. [Member 3] - [ID]4. [Member 4] - [ID]5. [Member 5] - [ID]---**What we're building:** A system that can detect lane lines on roads using traditional image processing techniques. We'll use edge detection, Hough transform, and some basic ML for line fitting.The idea is pretty straightforward - find the edges in the image, look for straight lines, and figure out which ones are the left and right lane markings.

## Contents1. Setting up libraries2. Loading our test images3. Image preprocessing (blur, edges, etc)4. Detecting lines with Hough Transform5. Separating left and right lanes6. Fitting the final lane lines7. Testing on different images8. What we learned

---## 1. Setting UpFirst, let's import everything we need. OpenCV does most of the heavy lifting for image processing, and we'll use sklearn for the RANSAC fitting later.

In [ ]:
# basic stuffimport numpy as npimport matplotlib.pyplot as pltimport cv2import os# for the ML partfrom sklearn.cluster import KMeansfrom sklearn.linear_model import RANSACRegressor, LinearRegressionfrom sklearn.preprocessing import StandardScaler# make plots look nicer%matplotlib inlineplt.rcParams['figure.figsize'] = [12, 8]print("opencv version:", cv2.__version__)print("ready to go!")

---## 2. Loading Test ImagesWe have a few road images to test our pipeline on. Let's load them and see what we're working with.

In [ ]:
# where our images are storedimg_folder = "images"out_folder = "output"# make output folder if it doesnt existif not os.path.exists(out_folder):    os.makedirs(out_folder)# grab all jpg/png filesimg_files = [f for f in os.listdir(img_folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]print(f"found {len(img_files)} images")# load them into a dictimages = {}for fname in img_files:    path = os.path.join(img_folder, fname)    img = cv2.imread(path)    if img is not None:        # opencv loads as BGR, convert to RGB for display        images[fname] = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)        print(f"  loaded {fname} - size: {img.shape[1]}x{img.shape[0]}")

In [ ]:
# lets see what these images look likefig, axes = plt.subplots(1, len(images), figsize=(15, 5))if len(images) == 1:    axes = [axes]for ax, (name, img) in zip(axes, images.items()):    ax.imshow(img)    ax.set_title(name)    ax.axis('off')plt.suptitle("Our Test Images", fontsize=14)plt.tight_layout()plt.show()

---## 3. PreprocessingBefore we can detect lanes, we need to clean up the image a bit. Here's what we'll do:1. **Convert to grayscale** - color doesn't really help us here, and grayscale is simpler to work with2. **Apply blur** - this smooths out noise that could mess up edge detection3. **Edge detection** - find where the intensity changes sharply (like lane markings!)4. **Mask the region of interest** - we only care about the road area, not the sky or treesLet's try each step and see what happens.

In [ ]:
# pick one image to experiment withtest_img = list(images.values())[0]test_name = list(images.keys())[0]print(f"using {test_name} for testing")# step 1: grayscalegray = cv2.cvtColor(test_img, cv2.COLOR_RGB2GRAY)# step 2: blur (kernel size 5 works well)blurred = cv2.GaussianBlur(gray, (5, 5), 0)# step 3: canny edge detection# the two numbers are thresholds - pixels with gradient above 150 are definitely edges,# below 50 are definitely not, and in between depends on neighborsedges = cv2.Canny(blurred, 50, 150)# show the progressionfig, axes = plt.subplots(1, 4, figsize=(16, 4))axes[0].imshow(test_img)axes[0].set_title("original")axes[1].imshow(gray, cmap='gray')axes[1].set_title("grayscale")axes[2].imshow(blurred, cmap='gray')axes[2].set_title("after blur")axes[3].imshow(edges, cmap='gray')axes[3].set_title("edges (canny)")for ax in axes:    ax.axis('off')plt.tight_layout()plt.show()print("you can see the lane lines show up nicely in the edge image!")

### Region of Interest (ROI)The edge image has a lot of stuff we don't care about - trees, other cars, the horizon, etc. We only want to look at the road area in front of us.We'll create a trapezoid mask that covers roughly where the lanes should be.

In [ ]:
def get_roi_vertices(img_shape):    # creates a trapezoid mask for the road area    # might need to tweak these values for different camera angles    h, w = img_shape[:2]        # trapezoid corners    vertices = np.array([[        (int(w * 0.1), h),             # bottom left        (int(w * 0.45), int(h * 0.6)), # top left          (int(w * 0.55), int(h * 0.6)), # top right        (int(w * 0.95), h)             # bottom right    ]], dtype=np.int32)        return verticesdef apply_mask(img, vertices):    # apply the ROI mask to an image    mask = np.zeros_like(img)    cv2.fillPoly(mask, vertices, 255)    return cv2.bitwise_and(img, mask)# create and apply the maskroi_vertices = get_roi_vertices(edges.shape)masked_edges = apply_mask(edges, roi_vertices)# visualizefig, axes = plt.subplots(1, 3, figsize=(15, 5))# show ROI on original imageimg_with_roi = test_img.copy()cv2.polylines(img_with_roi, roi_vertices, True, (255, 0, 0), 3)axes[0].imshow(img_with_roi)axes[0].set_title("ROI region (red outline)")axes[1].imshow(edges, cmap='gray')axes[1].set_title("all edges")axes[2].imshow(masked_edges, cmap='gray')axes[2].set_title("edges in ROI only")for ax in axes:    ax.axis('off')plt.tight_layout()plt.show()print("much cleaner! now we only have edges from the road area")

---## 4. Detecting Lines with Hough TransformNow comes the interesting part. We have edge pixels, but we need to find which ones form straight lines.The Hough Transform is a clever technique for this. The basic idea:- Any line can be written as: rho = x*cos(theta) + y*sin(theta)- For each edge pixel, we calculate all possible (rho, theta) pairs- Lines show up as peaks in this "parameter space"OpenCV has a probabilistic version (HoughLinesP) that's faster and gives us line segments directly.

In [ ]:
# apply hough transform# parameters took some trial and error to get rightlines = cv2.HoughLinesP(    masked_edges,    rho=1,              # distance resolution (1 pixel)    theta=np.pi/180,    # angle resolution (1 degree)    threshold=30,       # min votes to count as line    minLineLength=40,   # ignore short segments    maxLineGap=100      # connect segments with small gaps)print(f"found {len(lines) if lines is not None else 0} line segments")# draw all detected lines on the imagelines_img = test_img.copy()if lines is not None:    for line in lines:        x1, y1, x2, y2 = line[0]        cv2.line(lines_img, (x1, y1), (x2, y2), (255, 0, 0), 2)fig, axes = plt.subplots(1, 2, figsize=(14, 6))axes[0].imshow(masked_edges, cmap='gray')axes[0].set_title("edge image (input to hough)")axes[1].imshow(lines_img)axes[1].set_title(f"detected lines ({len(lines) if lines is not None else 0} segments)")for ax in axes:    ax.axis('off')plt.tight_layout()plt.show()

---## 5. Separating Left and Right LanesWe've got a bunch of line segments, but they're all mixed together. We need to figure out which ones belong to the left lane and which to the right.Here's the trick: **left lane lines have negative slope, right lane lines have positive slope** (in image coordinates where y increases downward).We'll also throw away any lines that are too horizontal - those are probably not lane markings.

In [ ]:
def classify_lines(lines, slope_thresh=0.3):    # separate lines into left lane, right lane, or junk    left = []    right = []        if lines is None:        return left, right        for line in lines:        x1, y1, x2, y2 = line[0]                # avoid division by zero        if x2 == x1:            continue                    slope = (y2 - y1) / (x2 - x1)        intercept = y1 - slope * x1                # classify based on slope        if slope < -slope_thresh:            left.append((slope, intercept, line[0]))        elif slope > slope_thresh:            right.append((slope, intercept, line[0]))        # else: too horizontal, ignore it        return left, rightleft_lines, right_lines = classify_lines(lines)print(f"left lane: {len(left_lines)} segments")print(f"right lane: {len(right_lines)} segments")# visualize the classificationfig, axes = plt.subplots(1, 3, figsize=(15, 5))# left lane (blue)left_img = test_img.copy()for _, _, (x1, y1, x2, y2) in left_lines:    cv2.line(left_img, (x1, y1), (x2, y2), (0, 0, 255), 2)axes[0].imshow(left_img)axes[0].set_title(f"left lane segments ({len(left_lines)})")# right lane (green)right_img = test_img.copy()for _, _, (x1, y1, x2, y2) in right_lines:    cv2.line(right_img, (x1, y1), (x2, y2), (0, 255, 0), 2)axes[1].imshow(right_img)axes[1].set_title(f"right lane segments ({len(right_lines)})")# bothboth_img = test_img.copy()for _, _, (x1, y1, x2, y2) in left_lines:    cv2.line(both_img, (x1, y1), (x2, y2), (0, 0, 255), 2)for _, _, (x1, y1, x2, y2) in right_lines:    cv2.line(both_img, (x1, y1), (x2, y2), (0, 255, 0), 2)axes[2].imshow(both_img)axes[2].set_title("both lanes classified")for ax in axes:    ax.axis('off')plt.tight_layout()plt.show()

---## 6. Fitting the Final Lane LinesWe have multiple line segments for each lane, but we want just ONE line per lane. This is where we use some ML techniques.We tried three different approaches:### Method 1: Simple AveragingJust take the average slope and intercept of all segments. Simple but can be thrown off by outliers.### Method 2: RANSACThis is more robust - it randomly samples points, fits a line, and keeps the one that has the most "inliers" (points close to the line). Outliers don't affect the result much.### Method 3: K-MeansCluster the lines in (slope, intercept) space and use the cluster center. Kind of overkill for this but interesting to try.

In [ ]:
def fit_line_average(line_data):    # simple averaging - just take mean of slopes and intercepts    if not line_data:        return None, None        slopes = [d[0] for d in line_data]    intercepts = [d[1] for d in line_data]        return np.mean(slopes), np.mean(intercepts)def fit_line_ransac(line_data):    # RANSAC fitting - more robust to outliers    if len(line_data) < 2:        return fit_line_average(line_data)        # collect all the points from line segments    points = []    for slope, intercept, (x1, y1, x2, y2) in line_data:        points.append([x1, y1])        points.append([x2, y2])        points = np.array(points)    X = points[:, 0].reshape(-1, 1)    y = points[:, 1]        try:        ransac = RANSACRegressor(            estimator=LinearRegression(),            min_samples=2,            residual_threshold=10.0,            random_state=42        )        ransac.fit(X, y)        slope = ransac.estimator_.coef_[0]        intercept = ransac.estimator_.intercept_        return slope, intercept    except:        # fall back to averaging if ransac fails        return fit_line_average(line_data)def fit_line_kmeans(line_data):    # kmeans clustering in slope-intercept space    if len(line_data) < 2:        return fit_line_average(line_data)        features = np.array([[d[0], d[1]] for d in line_data])        # normalize since slope and intercept have very different scales    scaler = StandardScaler()    features_scaled = scaler.fit_transform(features)        kmeans = KMeans(n_clusters=1, random_state=42, n_init=10)    kmeans.fit(features_scaled)        center = scaler.inverse_transform([kmeans.cluster_centers_[0]])[0]    return center[0], center[1]# try all three methodsprint("Fitting results:")print("-" * 40)for name, fit_func in [("Average", fit_line_average),                         ("RANSAC", fit_line_ransac),                         ("K-Means", fit_line_kmeans)]:    left_slope, left_int = fit_func(left_lines)    right_slope, right_int = fit_func(right_lines)        print(f"{name}:")    if left_slope:        print(f"  Left:  slope={left_slope:.3f}, intercept={left_int:.1f}")    if right_slope:        print(f"  Right: slope={right_slope:.3f}, intercept={right_int:.1f}")    print()

In [ ]:
def draw_lane_line(img, slope, intercept, color, thickness=8):    # draw a line from slope-intercept form    if slope is None or slope == 0:        return img        h = img.shape[0]    y1 = h  # bottom of image    y2 = int(h * 0.6)  # top of ROI        x1 = int((y1 - intercept) / slope)    x2 = int((y2 - intercept) / slope)        cv2.line(img, (x1, y1), (x2, y2), color, thickness)    return img# compare the three methods visuallyfig, axes = plt.subplots(1, 3, figsize=(15, 5))methods = [    ("Average", fit_line_average),    ("RANSAC", fit_line_ransac),    ("K-Means", fit_line_kmeans)]for ax, (name, fit_func) in zip(axes, methods):    result = test_img.copy()        left_slope, left_int = fit_func(left_lines)    right_slope, right_int = fit_func(right_lines)        draw_lane_line(result, left_slope, left_int, (255, 0, 0))  # blue for left    draw_lane_line(result, right_slope, right_int, (0, 255, 0))  # green for right        ax.imshow(result)    ax.set_title(f"{name} method")    ax.axis('off')plt.suptitle("Comparison of Line Fitting Methods", fontsize=14)plt.tight_layout()plt.show()print("All three give pretty similar results on clean images.")print("RANSAC would be better if there were more outliers/noise.")

---## 7. Putting It All TogetherLet's wrap everything into a single function and test it on all our images.

In [ ]:
def detect_lanes(image, method='ransac'):    '''    Complete lane detection pipeline        Steps:    1. Convert to grayscale    2. Apply gaussian blur    3. Detect edges with canny    4. Mask region of interest    5. Find lines with hough transform    6. Classify into left/right by slope    7. Fit final lines using specified method    8. Draw results    '''        # preprocessing    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)    blurred = cv2.GaussianBlur(gray, (5, 5), 0)    edges = cv2.Canny(blurred, 50, 150)        # ROI mask    roi_verts = get_roi_vertices(edges.shape)    masked = apply_mask(edges, roi_verts)        # hough transform    lines = cv2.HoughLinesP(masked, 1, np.pi/180, 30,                             minLineLength=40, maxLineGap=100)        # classify by slope    left, right = classify_lines(lines)        # fit lines    if method == 'average':        fit_func = fit_line_average    elif method == 'ransac':        fit_func = fit_line_ransac    else:        fit_func = fit_line_kmeans        left_slope, left_int = fit_func(left)    right_slope, right_int = fit_func(right)        # draw result    result = image.copy()        # draw filled polygon between lanes (looks nice)    if left_slope and right_slope:        h = image.shape[0]        y_top = int(h * 0.6)        y_bot = h                pts = np.array([            [int((y_bot - left_int) / left_slope), y_bot],            [int((y_top - left_int) / left_slope), y_top],            [int((y_top - right_int) / right_slope), y_top],            [int((y_bot - right_int) / right_slope), y_bot]        ], np.int32)                overlay = result.copy()        cv2.fillPoly(overlay, [pts], (0, 255, 0))        result = cv2.addWeighted(overlay, 0.3, result, 0.7, 0)        # draw lane lines    draw_lane_line(result, left_slope, left_int, (255, 0, 0), 8)    draw_lane_line(result, right_slope, right_int, (0, 255, 0), 8)        return result# test on all imagesprint("Processing all images...")print("=" * 50)fig, axes = plt.subplots(len(images), 2, figsize=(12, 5*len(images)))if len(images) == 1:    axes = axes.reshape(1, -1)for idx, (name, img) in enumerate(images.items()):    result = detect_lanes(img, method='ransac')        axes[idx, 0].imshow(img)    axes[idx, 0].set_title(f"Original: {name}")    axes[idx, 0].axis('off')        axes[idx, 1].imshow(result)    axes[idx, 1].set_title("Detected Lanes")    axes[idx, 1].axis('off')        # save result    out_path = os.path.join(out_folder, f"result_{name}")    cv2.imwrite(out_path, cv2.cvtColor(result, cv2.COLOR_RGB2BGR))    print(f"Processed {name} -> saved to {out_path}")plt.tight_layout()plt.show()print("=" * 50)print("Done! Results saved to output folder.")

---## 8. Analysis and Discussion### What worked well:- The pipeline works pretty reliably on clear road images with visible lane markings- RANSAC helps when there are some noisy line segments- The ROI mask is crucial - without it we'd pick up all sorts of irrelevant edges### Parameters we tuned:| Parameter | Value | Why ||-----------|-------|-----|| Blur kernel | 5x5 | Removes noise without losing too much detail || Canny low threshold | 50 | Catches faint lane markings || Canny high threshold | 150 | Standard 3:1 ratio with low threshold || Hough threshold | 30 | Balanced - not too many false positives || Min line length | 40px | Filters out tiny noise segments || Max line gap | 100px | Connects dashed lane markings || Slope threshold | 0.3 | Ignores near-horizontal lines |### Limitations we noticed:1. **Curved roads** - our approach assumes straight lines, so curves don't work well2. **Shadows** - strong shadows can create false edges that look like lanes3. **Faded markings** - if the paint is worn, edge detection might miss it4. **Other vehicles** - cars in front can block the view of lanes### Ideas for improvement:- Use polynomial fitting instead of lines for curved roads- Add color filtering to specifically look for white/yellow- Use temporal smoothing for video (average across frames)- Could try deep learning but that's a whole different approach

In [ ]:
# let's look at some statsprint("Detection Statistics")print("=" * 50)for name, img in images.items():    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)    blurred = cv2.GaussianBlur(gray, (5, 5), 0)    edges = cv2.Canny(blurred, 50, 150)    roi_verts = get_roi_vertices(edges.shape)    masked = apply_mask(edges, roi_verts)    lines = cv2.HoughLinesP(masked, 1, np.pi/180, 30, minLineLength=40, maxLineGap=100)    left, right = classify_lines(lines)        print(f"\n{name}:")    print(f"  Total edge pixels: {np.sum(edges > 0)}")    print(f"  Edge pixels in ROI: {np.sum(masked > 0)}")    print(f"  Line segments found: {len(lines) if lines is not None else 0}")    print(f"  Left lane segments: {len(left)}")    print(f"  Right lane segments: {len(right)}")    print(f"  Detection: {'SUCCESS' if left and right else 'PARTIAL' if left or right else 'FAILED'}")

---## ConclusionWe built a working lane detection system using classical computer vision techniques:1. **Preprocessing**: Grayscale + Gaussian blur to clean up the image2. **Edge Detection**: Canny algorithm to find intensity changes3. **ROI Masking**: Focus only on the road area4. **Line Detection**: Hough Transform to find straight lines5. **Classification**: Separate left/right lanes by slope6. **Line Fitting**: RANSAC for robust final line estimationThe system works well on clear road images but has limitations with curves, shadows, and occlusions. For a production system, you'd probably want to combine this with deep learning approaches, but for understanding the fundamentals of computer vision, this classical approach teaches a lot about how image processing works.---### References- OpenCV documentation: https://docs.opencv.org/- Canny Edge Detection: Canny, J. (1986). A computational approach to edge detection- Hough Transform: Duda, R. O., & Hart, P. E. (1972). Use of the Hough transformation to detect lines and curves in pictures

In [ ]:
# final summaryprint("=" * 60)print("LANE DETECTION - SUMMARY")print("=" * 60)print(f"Images processed: {len(images)}")print(f"Method used: RANSAC (robust fitting)")print(f"Output saved to: {out_folder}/")print()print("Parameters:")print("  Canny thresholds: 50, 150")print("  Hough: threshold=30, minLen=40, maxGap=100")print("  Slope threshold: 0.3")print("=" * 60)print("\nAssignment complete!")